# PS3 Rail Corrugation — CatBoost `r3_cat_sqrt_random2`

Raw Train files → vibration/shock features → 15 grouped CV fits (seeds 17, 29, 43 × 5 folds) → CatBoost. The printed evaluation metric is the task's three-class macro F1.

In [1]:
from pathlib import Path
import hashlib
import os
import numpy as np
import pandas as pd
from scipy.signal import welch
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from catboost import CatBoostClassifier

def find_data():
    override = os.environ.get('PS3_ROOT')
    candidates = []
    if override:
        root = Path(override)
        candidates += [root, root / '02_Datasets' / 'Rail_Corrugation']
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates += [base / 'data', base / 'Rail_Corrugation' / 'data',
                       base / 'NebulaX-Hackathon-ProblemStatement' / 'PS3' / '02_Datasets' / 'Rail_Corrugation']
    for candidate in candidates:
        if (candidate / 'Train').is_dir() and (candidate / 'Train_Labels.csv').is_file():
            return candidate
    raise FileNotFoundError('Set PS3_ROOT or run this notebook from the repository')

DATA = find_data()
BANDS = ((0, 100), (100, 300), (300, 800), (800, 1600), (1600, 3000), (3000, 5000))
LABELS = np.asarray(['Normal', 'Side I', 'Side II'])
print('Rail data:', DATA)

ModuleNotFoundError: No module named 'catboost'

In [ ]:
def extract(path):
    frame = pd.read_csv(path, header=0, dtype=np.float64)
    if frame.shape != (10000, 129) or frame.columns[0] != 'Rotating speed':
        raise ValueError(f'Unexpected shape/header: {path.name}: {frame.shape}')
    expected = ['Rotating speed'] + [
        f'{kind} of bearing in position {position} of car {car}'
        for car in range(1, 9) for position in range(1, 9)
        for kind in ('Vibration', 'Shock')]
    if list(frame.columns) != expected:
        raise ValueError(f'Channel order mismatch: {path.name}')
    values = frame.to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f'Nonfinite data: {path.name}')
    x = values[:, 1:]
    rms, std, absx = np.sqrt(np.mean(x * x, axis=0)), x.std(axis=0), np.abs(x)
    quantiles = np.quantile(absx, (0.5, 0.95, 0.99), axis=0)
    centered = x - x.mean(axis=0)
    kurt = np.mean(centered ** 4, axis=0) / np.maximum(std ** 4, 1e-24)
    crest = np.max(absx, axis=0) / np.maximum(rms, 1e-12)
    freq, density = welch(x, fs=10000.0, window='hann', nperseg=1024,
                          noverlap=512, detrend='constant', scaling='density', axis=0)
    power = density * (freq[1] - freq[0])
    total = power.sum(axis=0)
    shape = power / np.maximum(total, 1e-24)
    channel_stats = {
        'rms': rms, 'std': std, 'abs_q50': quantiles[0], 'abs_q95': quantiles[1],
        'abs_q99': quantiles[2], 'crest': crest, 'kurtosis': kurt,
        'spectral_centroid_hz': (shape * freq[:, None]).sum(axis=0),
        'spectral_entropy': -(shape * np.log(np.maximum(shape, 1e-24))).sum(axis=0) / np.log(len(freq)),
    }
    for low, high in BANDS:
        mask = (freq >= low) & (freq <= high if high == 5000 else freq < high)
        band_power = power[mask].sum(axis=0)
        channel_stats[f'log1p_band_power_{low}_{high}hz'] = np.log1p(band_power)
        channel_stats[f'band_fraction_{low}_{high}hz'] = band_power / np.maximum(total, 1e-24)
    result = {}
    for stat, values in channel_stats.items():
        array = values.reshape(8, 8, 2)
        for modality, name in enumerate(('vibration', 'shock')):
            for side in (0, 1):
                selected = array[:, side::2, modality]
                prefix = f'{name}_side{side + 1}_{stat}'
                result[f'{prefix}_mean'] = float(selected.mean())
                result[f'{prefix}_max'] = float(selected.max())
                for car in range(8):
                    result[f'car{car + 1}_{prefix}_mean'] = float(selected[car].mean())
            side_values = [float(array[:, side::2, modality].mean()) for side in (0, 1)]
            result[f'{name}_{stat}_side_difference'] = side_values[0] - side_values[1]
            result[f'{name}_{stat}_normalized_side_difference'] = (side_values[0] - side_values[1]) / (abs(side_values[0]) + abs(side_values[1]) + 1e-12)
    if not np.isfinite(list(result.values())).all():
        raise ValueError(f'Nonfinite features: {path.name}')
    return result

paths = sorted((DATA / 'Train').glob('*.csv'), key=lambda p: int(p.stem.removeprefix('Train')))
label_table = pd.read_csv(DATA / 'Train_Labels.csv').set_index('filename')
X_rows = [extract(p) for p in paths]
feature_names = list(X_rows[0])
X = np.asarray([[row[name] for name in feature_names] for row in X_rows], dtype=float)
label_to_int = {label: i for i, label in enumerate(LABELS)}
y = np.asarray([label_to_int[label_table.loc[p.name, 'label']] for p in paths], dtype=int)
groups = np.asarray([hashlib.sha256(p.read_bytes()).hexdigest() for p in paths])
global_ix = np.asarray([i for i, name in enumerate(feature_names) if not name.startswith('car')])
assert X.shape[1] == 924 and len(global_ix) == 252
print(f'files={len(paths)}, features={X.shape[1]}, active global features={len(global_ix)}')

In [ ]:
def make_model(seed):
    return CatBoostClassifier(
        auto_class_weights='SqrtBalanced', loss_function='MultiClass',
        bootstrap_type='Bernoulli', subsample=.85, random_strength=2.0,
        border_count=64, thread_count=2, verbose=False, allow_writing_files=False,
        iterations=500, depth=3, learning_rate=.025, l2_leaf_reg=8.0,
        random_seed=seed)

seeds = (17, 29, 43)
oof_probability_sum = np.zeros((len(y), 3), dtype=float)
oof_count = np.zeros(len(y), dtype=int)
fold_train_scores = []
for seed in seeds:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (fitting, validation) in enumerate(splitter.split(X, y, groups), 1):
        fitted = make_model(seed).fit(X[fitting][:, global_ix], y[fitting])
        train_pred = fitted.predict(X[fitting][:, global_ix]).reshape(-1).astype(int)
        validation_probability = fitted.predict_proba(X[validation][:, global_ix])
        oof_probability_sum[validation] += validation_probability
        oof_count[validation] += 1
        fold_train_scores.append(f1_score(y[fitting], train_pred, labels=[0, 1, 2], average='macro', zero_division=0))

assert np.all(oof_count == len(seeds))
oof_pred = np.argmax(oof_probability_sum / oof_count[:, None], axis=1)
evaluation_score = f1_score(y, oof_pred, labels=[0, 1, 2], average='macro', zero_division=0)

final_model = make_model(17).fit(X[:, global_ix], y)
train_pred = final_model.predict(X[:, global_ix]).reshape(-1).astype(int)
train_score = f1_score(y, train_pred, labels=[0, 1, 2], average='macro', zero_division=0)
print(f'Train macro F1 (all-Train fit, optimistic): {train_score:.6f}')
print(f'Evaluation macro F1 (15-fold grouped OOF, mean probabilities): {evaluation_score:.6f}')
print(f'Mean fold Train macro F1: {np.mean(fold_train_scores):.6f}')